# Prompt Management with LangSmith

This notebook demonstrates **prompt management** using LangSmith for a stock price assistant chatbot.

## What you will learn

| Concept | Description |
|---------|-------------|
| **Template** | Prompts in LangSmith are templates with variables (e.g. `{user_tickers}`) that get filled at runtime |
| **Messages List** | A chat prompt is a structured **list** of message templates, not a flat string |
| **Role** | Each message carries a **role** (`system`, `human`, `ai`) that shapes how the LLM interprets it |

## Project

A chatbot that checks stock prices using Yahoo Finance, personalized with the user's preferred ticker list.

## Setup

1. Run the setup cell below first — all other cells depend on it.
2. Set `MODEL_PROVIDER = "openai"` or `"gemini"`.
3. You need `LANGCHAIN_API_KEY` in your `.env` file for LangSmith access.
4. If packages are missing, run in your terminal:
   ```bash
   uv add langsmith yfinance
   ```

In [1]:
import os
import logging

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langsmith import Client
import yfinance as yf

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_PROVIDER = "openai"  # "gemini" | "openai"

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")

assert LANGCHAIN_API_KEY, "LANGCHAIN_API_KEY not found — add it to your .env file"

# ── Enable LangSmith tracing (observability for free) ──────────────────────────
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "stock-price-assistant"

# ── Shared LLM ────────────────────────────────────────────────────────────────
if MODEL_PROVIDER == "openai":
    assert OPENAI_API_KEY, "OPENAI_API_KEY not found — check your .env file"
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
else:
    assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-lite",
        google_api_key=GEMINI_API_KEY,
        temperature=0,
    )

logger.info("Setup complete — LLM provider: %s", MODEL_PROVIDER)

INFO | Setup complete — LLM provider: openai


## Stock Price Tool

Define a tool that fetches current stock prices using Yahoo Finance (`yfinance`).
The agent will call this tool when the user asks about stock prices.

In [2]:
@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a given ticker symbol.

    Args:
        ticker: Stock ticker symbol (e.g., AAPL, GOOGL, TSLA)
    Returns:
        Current stock price information including daily change
    """
    logger.info("get_stock_price called | ticker=%s", ticker)
    stock = yf.Ticker(ticker)
    hist = stock.history(period="1d")
    if hist.empty:
        return f"Could not fetch price for {ticker}"
    price = hist["Close"].iloc[-1]
    prev_close = stock.info.get("previousClose")
    result = f"{ticker}: ${price:.2f}"
    if prev_close:
        change = price - prev_close
        pct = (change / prev_close) * 100
        result += f" (change: {change:+.2f}, {pct:+.2f}%)"
    logger.info("get_stock_price result: %s", result)
    return result


logger.info("Stock price tool ready")

INFO | Stock price tool ready


---

## 1. Template — Pull Prompt from LangSmith

A **prompt template** is a reusable prompt with **variables** (placeholders like `{user_tickers}`)
that get filled with real values at runtime.

In LangSmith, you create and version prompts in the UI, then pull them in code:

```python
from langsmith import Client
client = Client()
prompt = client.pull_prompt("my-prompt-name")
```

The returned object is a `ChatPromptTemplate` — LangChain's standard prompt class.

> **Prerequisite:** Create the prompt in LangSmith before running the next cell.
> Update `PROMPT_NAME` below to match your prompt's name.

In [4]:
# ── Pull prompt from LangSmith ─────────────────────────────────────────────────
PROMPT_NAME = "stock-price-assistant"  # ← replace with your prompt name

client = Client()
prompt = client.pull_prompt(PROMPT_NAME)

# The prompt is a TEMPLATE — it has placeholders that we fill at runtime
print(f"Type       : {type(prompt).__name__}")
print(f"Variables  : {prompt.input_variables}")

Type       : ChatPromptTemplate
Variables  : ['question']


---

## 2. Messages List — A Prompt is a List of Messages

Unlike a simple string prompt, a **chat prompt** is a structured **list of messages**.
Each entry in the list is a message template that becomes a real message when formatted.

```python
prompt.messages  # → [SystemMessagePromptTemplate, HumanMessagePromptTemplate, ...]
```

This matters because:
- Each message can have its own template variables
- The **order** of messages matters — the LLM reads them top to bottom
- You can mix system instructions, example conversations, and user input in one prompt

In [5]:
# ── Inspect the messages list ──────────────────────────────────────────────────
print(f"Number of messages in prompt: {len(prompt.messages)}\n")

for i, msg_template in enumerate(prompt.messages):
    cls_name = type(msg_template).__name__
    print(f"  [{i}] {cls_name}")

    if hasattr(msg_template, "prompt") and hasattr(msg_template.prompt, "template"):
        template_text = msg_template.prompt.template
        preview = template_text[:120] + ("..." if len(template_text) > 120 else "")
        print(f"      Template: \"{preview}\"")
    elif hasattr(msg_template, "variable_name"):
        print(f"      Placeholder variable: {msg_template.variable_name}")
    print()

Number of messages in prompt: 2

  [0] SystemMessagePromptTemplate
      Template: "Answer user's question"

  [1] HumanMessagePromptTemplate
      Template: "{question}"



---

## 3. Roles — Each Message Has a Role

Every message in a chat prompt has a **role** that tells the LLM how to interpret it:

| Role | Purpose | Example |
|------|---------|--------|
| **system** | Sets the agent's behavior, personality, and constraints | *"You are a stock assistant…"* |
| **human** | Represents user input | *"What's the price of AAPL?"* |
| **ai** | Represents assistant responses (useful for few-shot examples) | *"AAPL is currently at $150.00"* |

The **system** role is especially powerful — it shapes the entire conversation.
This is where we inject the user's preferred tickers via the `{user_tickers}` template variable.

In [6]:
# ── Show the role of each message ──────────────────────────────────────────────
ROLE_MAP = {
    "SystemMessagePromptTemplate": "system",
    "HumanMessagePromptTemplate": "human",
    "AIMessagePromptTemplate": "ai",
    "MessagesPlaceholder": "placeholder",
}

print("Message roles in this prompt:\n")
for i, msg_template in enumerate(prompt.messages):
    cls_name = type(msg_template).__name__
    role = ROLE_MAP.get(cls_name, cls_name)
    print(f"  [{i}] Role: {role:<12} | Type: {cls_name}")

Message roles in this prompt:

  [0] Role: system       | Type: SystemMessagePromptTemplate
  [1] Role: human        | Type: HumanMessagePromptTemplate


---

## Format the Template

Now we **format** the template by filling in the variables with real values.
The user's preferred tickers are injected into the `{user_tickers}` variable.

After formatting, each template becomes a real message (`SystemMessage`, `HumanMessage`, etc.)
with concrete content — ready to send to the LLM.

In [7]:
# ── User's preferred tickers ───────────────────────────────────────────────────
USER_TICKERS = ["AAPL", "GOOGL", "TSLA"]

# Format the system message template with the user's tickers
system_template = prompt.messages[0]
formatted_system = system_template.format(user_tickers=", ".join(USER_TICKERS))

print(f"Type   : {type(formatted_system).__name__}")
print(f"Role   : {formatted_system.type}")
print(f"Content:\n")
print(formatted_system.content)

Type   : SystemMessage
Role   : system
Content:

Answer user's question


---

## Build the Agent

Wire everything together:
1. The **formatted system prompt** (with tickers injected) tells the agent who it is
2. The **stock price tool** gives the agent the ability to look up real prices
3. `create_agent` builds a ReAct loop that reasons and acts autonomously

In [8]:
# ── Build the agent with the formatted prompt ─────────────────────────────────
agent = create_agent(
    model=llm,
    tools=[get_stock_price],
    system_prompt=formatted_system.content,
) # we can create an agent without a system prompt then add full list of messages later

logger.info("Agent created — prompt injected, tools: %s", [get_stock_price.name])

INFO | Agent created — prompt injected, tools: ['get_stock_price']


---

## Test the Agent

Ask the agent about the user's preferred stocks.
Because the system prompt contains the ticker list, the agent knows which stocks to check.

In [9]:
# ── Test 1: Ask about preferred stocks ─────────────────────────────────────────
result = agent.invoke(
    {"messages": [HumanMessage(content="How are my preferred stocks doing today?")]}
)

print(result["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Please provide me with the ticker symbols of your preferred stocks, and I'll check their current stock prices for you.


In [10]:
# ── Test 2: Ask about a specific stock ─────────────────────────────────────────
result2 = agent.invoke(
    {"messages": [HumanMessage(content="What is the current price of NVDA?")]}
)

print(result2["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | get_stock_price called | ticker=NVDA
INFO | get_stock_price result: NVDA: $167.52 (change: -3.72, -2.17%)
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The current price of NVDA (NVIDIA Corporation) is $167.52, with a change of -$3.72, which is a decrease of 2.17%.


---

## Wrap-up

### What we covered

| Concept | What you saw |
|---------|-------------|
| **Template** | `prompt.input_variables` — the prompt has `{user_tickers}` filled at runtime |
| **Messages List** | `prompt.messages` — a prompt is a list of message templates, not a flat string |
| **Role** | Each message has a role (`system`, `human`, `ai`) that shapes LLM behavior |

### Key takeaways

- **LangSmith** stores prompts centrally — update them in the UI without redeploying code
- **Templates** let you inject user-specific data (like preferred tickers) at runtime
- **Roles** control how the LLM interprets each message — the `system` role is the most powerful lever
- **Observability** is automatic — every agent run is traced in LangSmith (check your project dashboard)